In [1]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'

In [2]:
import numpy as np
import pandas as pd
import pickle
import warnings
warnings.filterwarnings('ignore')

In [3]:
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import KFold, cross_val_score
from sklearn.svm import SVR
import tensorflow as tf
from tensorflow.keras.models import load_model


In [4]:
tf.get_logger().setLevel('ERROR')

In [6]:
# LOAD DATA & MODELS
df_train = pd.read_csv("train_grocery_inventory.csv", parse_dates=["Date"])
df_test  = pd.read_csv("test_grocery_inventory.csv",  parse_dates=["Date"])

FEATURE_COLS = [c for c in df_train.columns
                if c not in ['Date', 'Store ID', 'Product ID', 'Units Sold']]
TARGET_COL = 'Units Sold'

X_train = df_train[FEATURE_COLS].values
y_train = df_train[TARGET_COL].values
X_test  = df_test[FEATURE_COLS].values
y_test  = df_test[TARGET_COL].values

lstm_model = load_model("lstm_best_model.keras")
with open("svr_model.pkl", "rb") as f:
    svr_model = pickle.load(f)

print(" Models loaded: LSTM + SVR")

 Models loaded: LSTM + SVR


In [7]:
# SEQUENCE BUILDER
SEQ_LEN = 14

def make_sequences(X, y, seq_len):
    Xs, ys = [], []
    for i in range(seq_len, len(X)):
        Xs.append(X[i - seq_len:i])
        ys.append(y[i])
    return np.array(Xs), np.array(ys)

X_train_seq, y_train_seq = make_sequences(X_train, y_train, SEQ_LEN)
X_test_seq,  y_test_seq  = make_sequences(X_test,  y_test,  SEQ_LEN)
X_test_svr  = X_test[SEQ_LEN:]

In [8]:
# METRICS 
def smape(actual, predicted):
    """Symmetric MAPE — robust to near-zero values."""
    return 100 * np.mean(
        2 * np.abs(predicted - actual) /
        (np.abs(actual) + np.abs(predicted) + 1e-8)
    )

def evaluate(name, y_true, y_pred):
    mape_val = smape(y_true, y_pred)
    rmse_val = np.sqrt(mean_squared_error(y_true, y_pred))
    acc      = max(0, 100 - mape_val)
    return {"Model": name, "sMAPE (%)": round(mape_val, 4),
            "RMSE": round(rmse_val, 6), "Accuracy (%)": round(acc, 2)}

In [9]:
# LSTM ON TEST SET
print("\n" + "-" * 60)
print("LSTM EVALUATION ON TEST SET")
print("-" * 60)
lstm_preds  = lstm_model.predict(X_test_seq, verbose=0).flatten()
lstm_metrics = evaluate("LSTM", y_test_seq, lstm_preds)
print(f"  sMAPE    : {lstm_metrics['sMAPE (%)']:.4f}%")
print(f"  RMSE     : {lstm_metrics['RMSE']:.6f}  (scaled 0–1)")
print(f"  Accuracy : {lstm_metrics['Accuracy (%)']:.2f}%")


------------------------------------------------------------
LSTM EVALUATION ON TEST SET
------------------------------------------------------------
  sMAPE    : 72.3491%
  RMSE     : 0.247473  (scaled 0–1)
  Accuracy : 27.65%


In [10]:
# SVR ON TEST SET
print("\n" + "-" * 60)
print("SVR EVALUATION ON TEST SET")
print("-" * 60)
svr_preds   = svr_model.predict(X_test_svr)
svr_metrics = evaluate("SVR", y_test_seq, svr_preds)
print(f"  sMAPE    : {svr_metrics['sMAPE (%)']:.4f}%")
print(f"  RMSE     : {svr_metrics['RMSE']:.6f}  (scaled 0–1)")
print(f"  Accuracy : {svr_metrics['Accuracy (%)']:.2f}%")


------------------------------------------------------------
SVR EVALUATION ON TEST SET
------------------------------------------------------------
  sMAPE    : 17.7027%
  RMSE     : 0.020455  (scaled 0–1)
  Accuracy : 82.30%


In [11]:
# CROSS-VALIDATION  (SVR, 5-fold, no shuffle)
print("\n" + "-" * 60)
print("CROSS-VALIDATION  (SVR · 5-Fold · time-series)")
print("-" * 60)
kf = KFold(n_splits=5, shuffle=False)
cv_scores = cross_val_score(
    SVR(kernel='rbf', C=100, epsilon=0.01, gamma='scale'),
    X_train, y_train,
    cv=kf,
    scoring='neg_mean_squared_error',
    n_jobs=-1
)
cv_rmse = np.sqrt(-cv_scores)
print(f"  Fold RMSE : {[round(v, 6) for v in cv_rmse]}")
print(f"  Mean RMSE : {cv_rmse.mean():.6f}")
print(f"  Std  RMSE : {cv_rmse.std():.6f}")


------------------------------------------------------------
CROSS-VALIDATION  (SVR · 5-Fold · time-series)
------------------------------------------------------------
  Fold RMSE : [np.float64(0.020176), np.float64(0.0202), np.float64(0.020585), np.float64(0.020334), np.float64(0.020796)]
  Mean RMSE : 0.020418
  Std  RMSE : 0.000238


In [15]:
# COMPARISON TABLE
print("\n" + "-" * 60)
print("MODEL COMPARISON")
print("-" * 60)
results_df = pd.DataFrame([lstm_metrics, svr_metrics])
results_df["CV RMSE (mean)"] = [None, round(cv_rmse.mean(), 6)]
results_df["CV RMSE (std)"]  = [None, round(cv_rmse.std(), 6)]
print(results_df.to_string(index=False))

best = results_df.sort_values("sMAPE (%)").iloc[0]["Model"]
print(f"\n  Best model (lowest sMAPE): {best}")
print(f"  LSTM performance is limited by short per-SKU history (~132 rows).")
print(f"  With longer real-world POS history, LSTM accuracy will improve.")


------------------------------------------------------------
MODEL COMPARISON
------------------------------------------------------------
Model  sMAPE (%)     RMSE  Accuracy (%)  CV RMSE (mean)  CV RMSE (std)
 LSTM    72.3491 0.247473         27.65             NaN            NaN
  SVR    17.7027 0.020455         82.30        0.020418       0.000238

  Best model (lowest sMAPE): SVR
  LSTM performance is limited by short per-SKU history (~132 rows).
  With longer real-world POS history, LSTM accuracy will improve.


In [14]:
# REORDER & EXPIRY FLAGS + SAVE
test_dates      = df_test["Date"].values[SEQ_LEN:]
test_store_ids  = df_test["Store ID"].values[SEQ_LEN:]
test_product_ids= df_test["Product ID"].values[SEQ_LEN:]
test_inventory  = df_test["Inventory Level"].values[SEQ_LEN:]

pred_df = pd.DataFrame({
    "Date":           test_dates,
    "Store ID":       test_store_ids,
    "Product ID":     test_product_ids,
    "Actual":         y_test_seq,
    "LSTM_Predicted": lstm_preds,
    "SVR_Predicted":  svr_preds,
    "Inventory_Level":test_inventory,
})

pred_df["Reorder_Alert"] = (
    pred_df["LSTM_Predicted"] > pred_df["Inventory_Level"] * 0.70
).astype(int)

pred_df["Expiry_Risk"] = (
    (pred_df["Inventory_Level"] > 0.60) &
    (pred_df["LSTM_Predicted"] < 0.30)
).astype(int)

pred_df.to_csv("validation_predictions.csv", index=False)
results_df.to_csv("validation_metrics.csv",  index=False)

print(f"\n Predictions saved     → validation_predictions.csv")
print(f" Metrics saved         → validation_metrics.csv")
print(f" Reorder alerts        : {pred_df['Reorder_Alert'].sum()} flagged")
print(f" Expiry risk flags     : {pred_df['Expiry_Risk'].sum()} flagged")


 Predictions saved     → validation_predictions.csv
 Metrics saved         → validation_metrics.csv
 Reorder alerts        : 1137 flagged
 Expiry risk flags     : 25 flagged
